# Day 13 — Confidence Intervals
**Dataset:** FC Lahore Lions (synthetic, seed=42)

Goal: build a 95% Confidence Interval for a team stat, understand what
"95% confident" really means, and see how sample size affects CI width.

## 1. Imports & simulate a season of matches

In [1]:
import numpy as np
from scipy import stats

np.random.seed(42)

n_matches = 38  # one football season
goals_scored = np.random.poisson(lam=1.6, size=n_matches)
possession = np.random.normal(loc=55, scale=6, size=n_matches)

print("Goals scored sample:", goals_scored)

Goals scored sample: [3 0 0 0 3 2 0 0 1 1 1 1 2 2 1 0 4 2 1 2 1 1 6 0 0 1 2 1 0 1 2 0 4 1 2 0 1
 3]


## 2. Confidence Interval — Goals per Match

In [2]:
mean_goals = np.mean(goals_scored)
std_goals = np.std(goals_scored, ddof=1)   # ddof=1 -> sample std, not population std
n = len(goals_scored)
se_goals = std_goals / np.sqrt(n)

ci_goals = stats.t.interval(0.95, df=n-1, loc=mean_goals, scale=se_goals)

print("Sample size:", n)
print("Mean goals/match:", round(mean_goals, 3))
print("Std dev:", round(std_goals, 3))
print("Standard Error:", round(se_goals, 4))
print("95% Confidence Interval:", (round(ci_goals[0], 3), round(ci_goals[1], 3)))

Sample size: 38
Mean goals/match: 1.368
Std dev: 1.364
Standard Error: 0.2213
95% Confidence Interval: (np.float64(0.92), np.float64(1.817))


**Reading it out loud:** We're 95% confident Lions' true long-run average
goals-per-match is somewhere in this range. Remember — this describes the
reliability of the *method* over repeated sampling, not a 95% chance for
this one specific interval.

## 3. Confidence Interval — Possession %

In [3]:
mean_poss = np.mean(possession)
std_poss = np.std(possession, ddof=1)
se_poss = std_poss / np.sqrt(n)

ci_poss = stats.t.interval(0.95, df=n-1, loc=mean_poss, scale=se_poss)

print("Mean possession %:", round(mean_poss, 3))
print("Std dev:", round(std_poss, 3))
print("Standard Error:", round(se_poss, 4))
print("95% Confidence Interval:", (round(ci_poss[0], 3), round(ci_poss[1], 3)))

Mean possession %: 54.372
Std dev: 5.301
Standard Error: 0.8599
95% Confidence Interval: (np.float64(52.63), np.float64(56.114))


## 4. z-interval vs t-interval comparison

In [4]:
z_ci = (mean_goals - 1.96 * se_goals, mean_goals + 1.96 * se_goals)

print("t-based 95% CI:", (round(ci_goals[0], 3), round(ci_goals[1], 3)))
print("z-based 95% CI:", (round(z_ci[0], 3), round(z_ci[1], 3)))

t-based 95% CI: (np.float64(0.92), np.float64(1.817))
z-based 95% CI: (np.float64(0.935), np.float64(1.802))


t and z are close here because n=38 is reasonably large — but t is the
more honest choice since we never actually know the true population std
dev. As a rule: **use t by default.**

## 5. Sample size effect — does more data narrow the CI?

In [5]:
np.random.seed(42)

for size in [10, 38, 100, 380]:
    sample = np.random.poisson(lam=1.6, size=size)
    m = np.mean(sample)
    s = np.std(sample, ddof=1)
    se = s / np.sqrt(size)
    ci = stats.t.interval(0.95, df=size-1, loc=m, scale=se)
    width = ci[1] - ci[0]
    print(f"n={size:>4} | mean={m:.3f} | CI=({ci[0]:.3f}, {ci[1]:.3f}) | width={width:.3f}")

n=  10 | mean=1.000 | CI=(0.108, 1.892) | width=1.784
n=  38 | mean=1.421 | CI=(0.971, 1.871) | width=0.899
n= 100 | mean=1.750 | CI=(1.472, 2.028) | width=0.557
n= 380 | mean=1.566 | CI=(1.433, 1.698) | width=0.265


**Takeaway:** as `n` goes up, the CI keeps getting narrower — but not in a
straight line. Because of the square root in the Standard Error formula,
you need **4x** the data to cut the CI width in **half**, not just 2x.

## 6. Practice task — e-commerce average purchase amount (roadmap task)

In [6]:
np.random.seed(42)

purchase_amounts = np.random.gamma(shape=2.0, scale=25.0, size=150)  # skewed, realistic spend data

mean_purchase = np.mean(purchase_amounts)
std_purchase = np.std(purchase_amounts, ddof=1)
n_purchase = len(purchase_amounts)
se_purchase = std_purchase / np.sqrt(n_purchase)

ci_purchase = stats.t.interval(0.95, df=n_purchase-1, loc=mean_purchase, scale=se_purchase)

print("Mean purchase amount:", round(mean_purchase, 2))
print("95% Confidence Interval:", (round(ci_purchase[0], 2), round(ci_purchase[1], 2)))

Mean purchase amount: 47.55
95% Confidence Interval: (np.float64(42.62), np.float64(52.48))


## 7. Summary

- CI = mean ± (critical value) × Standard Error
- Use **t-distribution** in almost all real cases (population std dev unknown)
- Bigger sample size → smaller SE → narrower CI
- "95% confident" = reliability of the method across repeated sampling,
  **not** a 95% chance for this one specific interval

**Quiz score: 9/10** — missed the classic "95% chance" wording trap once,
correctly self-corrected in later questions. No revisit flag needed.

**Next:** Day 14 — Covariance & Advanced Correlation Concepts